In [42]:
import csv
import os
import pyodbc # or your preferred DB connector

In [20]:
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=AshishPC;"
    "DATABASE=WideWorldImporters;"
    "Trusted_Connection=yes;"
)

In [21]:
cnxn = pyodbc.connect(connection_string)

In [29]:
cursor = cnxn.cursor()

In [43]:
# --- Configuration ---
# Set the output directory
OUTPUT_DIR = "c:/exported_tables"
DELIMITER = "~"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"Created output directory: {OUTPUT_DIR}")

# 2. Get the list of all base tables
print("Fetching list of all base tables...")
cursor.execute(
    "SELECT TABLE_SCHEMA, TABLE_NAME "
    "FROM information_schema.tables "
    "WHERE table_type = 'BASE TABLE'"
)
tables = cursor.fetchall()
print(f"Found {len(tables)} tables to export.")

# 3. Loop through each table and export its data
for schema_name, table_name in tables:
    full_table_name = f"[{schema_name}].[{table_name}]"
    
    # Define the output file name
    output_filename = os.path.join(OUTPUT_DIR, f"{schema_name}_{table_name}.csv")
    
    print(f"\nExporting data from: {full_table_name} to {output_filename}...")
    
    try:
        # A. Execute the query to select all data
        # Use brackets [] to handle spaces or special characters in names
        cursor.execute(f"SELECT * FROM {full_table_name}")
        
        # B. Get column names from the cursor description
        # This is crucial for the CSV header row
        column_names = [column[0] for column in cursor.description]
        
        # C. Fetch all rows
        rows = cursor.fetchall()
        
        # D. Write data to the CSV file
        with open(output_filename, 'w', newline='', encoding='utf-8') as f:
            # Tell the csv writer to use the '~' delimiter
            writer = csv.writer(f, delimiter=DELIMITER, quoting=csv.QUOTE_MINIMAL)
            
            # Write the header row
            writer.writerow(column_names)
            
            # Write the data rows
            writer.writerows(rows)
            
        print(f"Successfully exported {len(rows)} rows.")

    except Exception as e:
        print(f"!!! FAILED to export table {full_table_name}. Error: {e}")
        # Note: If you encounter the 'hierarchyid' error (-151), 
        # you will need to list columns explicitly here instead of using SELECT *.

# 4. Cleanup
# conn.close()
print("\nExport process finished.")

Created output directory: c:/exported_tables
Fetching list of all base tables...
Found 48 tables to export.

Exporting data from: [Warehouse].[Colors] to c:/exported_tables\Warehouse_Colors.csv...
Successfully exported 36 rows.

Exporting data from: [Warehouse].[Colors_Archive] to c:/exported_tables\Warehouse_Colors_Archive.csv...
Successfully exported 1 rows.

Exporting data from: [Sales].[OrderLines] to c:/exported_tables\Sales_OrderLines.csv...
Successfully exported 231412 rows.

Exporting data from: [Warehouse].[PackageTypes] to c:/exported_tables\Warehouse_PackageTypes.csv...
Successfully exported 14 rows.

Exporting data from: [Warehouse].[PackageTypes_Archive] to c:/exported_tables\Warehouse_PackageTypes_Archive.csv...
Successfully exported 0 rows.

Exporting data from: [Warehouse].[StockGroups] to c:/exported_tables\Warehouse_StockGroups.csv...
Successfully exported 10 rows.

Exporting data from: [Warehouse].[StockItemStockGroups] to c:/exported_tables\Warehouse_StockItemStockG

In [44]:
cursor.close()
cnxn.close()